# Conversation Generation

Stage 1 of the social-engineering detection project. This notebook runs
**threat** and **benign** generation as two separate runs with their own
outputs and their own settings.

Layout:
```
project_root/
├── conversation_generation/     <- this folder
│   ├── conversation_generation.py
│   ├── prompt_dimensions.json
│   ├── run_generation.ipynb     (main pipeline; you are here)
│   ├── dev.ipynb                (single-prompt / single-call testing)
│   └── .env                     (OPENAI_API_KEY=sk-...)
└── conversations/
    ├── threat_metadata.xlsx
    ├── threat_conversations.json
    ├── benign_metadata.xlsx
    └── benign_conversations.json
```

In [1]:
import conversation_generation as cg
from collections import Counter

DIMENSIONS_PATH = 'prompt_dimensions.json'
dimensions = cg.load_dimensions(DIMENSIONS_PATH)
print('Templates:        ', list(dimensions['prompt_templates'].keys()))
print('Scenarios:        ', list(dimensions['scenarios'].keys()))
print('Representatives:  ', list(dimensions['representatives'].keys()))
print('Threat callers:   ', list(dimensions['threat_caller_profile'].keys()))
print('Benign callers:   ', list(dimensions['benign_caller_profile'].keys()))
print('Benign context:   ', list(dimensions['benign_context_levels'].keys()))
print('Cialdini:         ', list(dimensions['cialdini_emphasis'].keys()))
print('Turn counts:      ', list(dimensions['turn_counts'].keys()))
print(f'Flavors:           {len(dimensions["flavors"])} indexed entries')

Templates:         ['threat', 'benign']
Scenarios:         ['credit_union']
Representatives:   ['overly_helpful', 'tired_but_professional', 'by_the_book']
Threat callers:    ['opportunistic_attacker']
Benign callers:    ['routine_balance_check', 'confused_first_time', 'frustrated_legitimate', 'chatty_regular', 'household_proxy', 'automated_question', 'recently_widowed', 'expat_abroad']
Benign context:    ['minimal', 'moderate', 'heavy']
Cialdini:          ['reciprocity', 'commitment_consistency', 'social_proof', 'authority', 'liking', 'scarcity', 'unity']
Turn counts:       ['standard']
Flavors:           32 indexed entries


---
## Threat run

Deterministic flavor subset: 32 flavors are sampled once (using the seed)
and reused for every dimension combination.

In [2]:
# ---- THREAT RUN CONFIG ----
THREAT_METADATA_PATH      = '../conversations/threat_metadata.xlsx'
THREAT_CONVERSATIONS_PATH = '../conversations/threat_conversations.json'

THREAT_MODEL        = 'gpt-5.4'
THREAT_TEMPERATURE  = 1
THREAT_TOP_P        = 1

THREAT_FLAVOR_COUNT    = 32
THREAT_FLAVOR_STRATEGY = cg.FLAVOR_DETERMINISTIC
THREAT_FLAVOR_SEED     = 42
THREAT_REPLICATES      = 1

THREAT_BATCH_SIZE  = 5
THREAT_FLUSH_EVERY = 20

THREAT_SMOKE_TEST_LIMIT = 2

In [3]:
threat_requests = cg.enumerate_requests(
    dimensions,
    template_key    = 'threat',
    flavor_count    = THREAT_FLAVOR_COUNT,
    flavor_strategy = THREAT_FLAVOR_STRATEGY,
    model           = THREAT_MODEL,
    temperature     = THREAT_TEMPERATURE,
    top_p           = THREAT_TOP_P,
    replicates      = THREAT_REPLICATES,
    flavor_seed     = THREAT_FLAVOR_SEED,
)
cg.attach_prompts(threat_requests, dimensions)
print(f'Threat requests: {len(threat_requests)}')

cg.write_metadata_xlsx(threat_requests, THREAT_METADATA_PATH)
print(f'Wrote: {THREAT_METADATA_PATH}')

Threat requests: 2016
Wrote: ../conversations/threat_metadata.xlsx


In [ ]:
# Smoke test
cg.run_generation_loop(
    metadata_path      = THREAT_METADATA_PATH,
    conversations_path = THREAT_CONVERSATIONS_PATH,
    max_requests       = THREAT_SMOKE_TEST_LIMIT,
    batch_size         = THREAT_BATCH_SIZE,
    flush_every        = THREAT_FLUSH_EVERY,
)

# Token usage from the rows just generated
rows = cg.read_metadata_xlsx(THREAT_METADATA_PATH)
succ = [r for r in rows if r['status'] == 'success']
if succ:
    in_tot  = sum(int(r['input_tokens']  or 0) for r in succ)
    out_tot = sum(int(r['output_tokens'] or 0) for r in succ)
    tot_tot = sum(int(r['total_tokens']  or 0) for r in succ)
    print(f'Threat smoke test - {len(succ)} successes')
    print(f'  total tokens:   input={in_tot:>8,} output={out_tot:>8,} total={tot_tot:>8,}')
    print(f'  per-conv mean:  input={in_tot//len(succ):>8,} output={out_tot//len(succ):>8,} total={tot_tot//len(succ):>8,}')

In [4]:
# Full threat run (resumable; skips successes, retries errors)
cg.run_generation_loop(
    metadata_path      = THREAT_METADATA_PATH,
    conversations_path = THREAT_CONVERSATIONS_PATH,
    max_requests       = None,
    batch_size         = THREAT_BATCH_SIZE,
    flush_every        = THREAT_FLUSH_EVERY,
)

Generating conversations:   0%|          | 0/1247 [00:00<?, ?conv/s]

---
## Benign run

Resampled flavor subset: 5 flavors are drawn fresh for each dimension
combination (still seeded for reproducibility across runs).

In [ ]:
# ---- BENIGN RUN CONFIG ----
BENIGN_METADATA_PATH      = '../conversations/benign_metadata.xlsx'
BENIGN_CONVERSATIONS_PATH = '../conversations/benign_conversations.json'

BENIGN_MODEL        = 'gpt-5.4'
BENIGN_TEMPERATURE  = 1
BENIGN_TOP_P        = 1

BENIGN_FLAVOR_COUNT    = 2
BENIGN_FLAVOR_STRATEGY = cg.FLAVOR_RESAMPLED
BENIGN_FLAVOR_SEED     = 42
BENIGN_REPLICATES      = 1

BENIGN_BATCH_SIZE  = 5
BENIGN_FLUSH_EVERY = 20

BENIGN_SMOKE_TEST_LIMIT = 2

In [ ]:
benign_requests = cg.enumerate_requests(
    dimensions,
    template_key    = 'benign',
    flavor_count    = BENIGN_FLAVOR_COUNT,
    flavor_strategy = BENIGN_FLAVOR_STRATEGY,
    model           = BENIGN_MODEL,
    temperature     = BENIGN_TEMPERATURE,
    top_p           = BENIGN_TOP_P,
    replicates      = BENIGN_REPLICATES,
    flavor_seed     = BENIGN_FLAVOR_SEED,
)
cg.attach_prompts(benign_requests, dimensions)
print(f'Benign requests: {len(benign_requests)}')

cg.write_metadata_xlsx(benign_requests, BENIGN_METADATA_PATH)
print(f'Wrote: {BENIGN_METADATA_PATH}')

In [ ]:
# Smoke test
cg.run_generation_loop(
    metadata_path      = BENIGN_METADATA_PATH,
    conversations_path = BENIGN_CONVERSATIONS_PATH,
    max_requests       = BENIGN_SMOKE_TEST_LIMIT,
    batch_size         = BENIGN_BATCH_SIZE,
    flush_every        = BENIGN_FLUSH_EVERY,
)

# Token usage from the rows just generated
rows = cg.read_metadata_xlsx(BENIGN_METADATA_PATH)
succ = [r for r in rows if r['status'] == 'success']
if succ:
    in_tot  = sum(int(r['input_tokens']  or 0) for r in succ)
    out_tot = sum(int(r['output_tokens'] or 0) for r in succ)
    tot_tot = sum(int(r['total_tokens']  or 0) for r in succ)
    print(f'Benign smoke test - {len(succ)} successes')
    print(f'  total tokens:   input={in_tot:>8,} output={out_tot:>8,} total={tot_tot:>8,}')
    print(f'  per-conv mean:  input={in_tot//len(succ):>8,} output={out_tot//len(succ):>8,} total={tot_tot//len(succ):>8,}')

In [ ]:
# Full benign run (resumable; skips successes, retries errors)
cg.run_generation_loop(
    metadata_path      = BENIGN_METADATA_PATH,
    conversations_path = BENIGN_CONVERSATIONS_PATH,
    max_requests       = None,
    batch_size         = BENIGN_BATCH_SIZE,
    flush_every        = BENIGN_FLUSH_EVERY,
)

---
## Status summary

In [ ]:
for label, path in [('threat', THREAT_METADATA_PATH), ('benign', BENIGN_METADATA_PATH)]:
    try:
        rows = cg.read_metadata_xlsx(path)
    except FileNotFoundError:
        print(f'{label}: not yet written')
        continue
    statuses = Counter(r['status'] for r in rows)
    print(f'{label}: {dict(statuses)}')